<a href="https://colab.research.google.com/github/LAB-FAM/nice-rag-project/blob/main/colab/data_ingestion_v4_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. INSTALL REQUIRED LIBRARIES
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr
!pip install langchain langchain-community langchain-experimental chromadb langchain-chroma unstructured[pdf] langchain-openai

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-up

In [1]:
import os
import re
import shutil
import getpass
from google.colab import drive
from langchain_community.document_loaders import DirectoryLoader, UnstructuredPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [2]:
# --- SET OPENAI API KEY ---
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

Enter your OpenAI API Key: ··········


In [3]:
# --- MOUNT GOOGLE DRIVE ---
print("[*] Mounting Google Drive...")
drive.mount('/content/drive')

[*] Mounting Google Drive...
Mounted at /content/drive


In [4]:
# --- FOLDER AND MODEL SETTINGS ---
BASE_DIR = "/content/drive/MyDrive/nice_rag_prod"
PDF_DOCS_DIR = f"{BASE_DIR}/pdfs"
DRIVE_DB_DIR = f"{BASE_DIR}/vector_db/methodology_db"
LOCAL_DB_DIR = "/content/local_vector_db/methodology_db"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"

In [5]:
# Create directories if they do not exist
os.makedirs(PDF_DOCS_DIR, exist_ok=True)

In [6]:
# --- DATA CLEANING REGEX PATTERNS ---
# --- NEW: Headers and Footers (PDF Ghosting Removal) ---
BOILERPLATE_PATTERNS = [
    r"Subject to Notice of rights.*",
    r"Return to recommendations",
    r"Why the committee made the recommendations",
    r"How the recommendations might affect practice",
    r"© NICE \d{4}\. All rights reserved\..*",
    r"Page\s+\d+\s+of\s+\d+",
    r"Hypertension in adults: diagnosis and management \(NG136\)",
    r"Type 2 diabetes in adults: management \(NG28\)",
    r"Overweight and obesity management \(NG246\)",
    r"Quality and Outcomes Framework guidance for \d{4}/\d{2}",
    r"© NHS England \d{4}"
]

# low-value section keywords adapted for chunk filtering
LOW_VALUE_KEYWORDS = [
    "table of contents",
    "contents",
    "your responsibility",
    "update information",
    "finding more information and committee details"
]

In [7]:
def clean_boilerplate_text(text: str) -> str:
    """Applies Regex patterns to remove noise from the extracted text."""

    if not text:
        return ""

    text = text.replace("\r", "\n")

    # Remove boilerplate
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    # Remove Table of Contents entries completely (removes the text, the dots, and the page numbers)
    # e.g., "General principles of care ................ 8" becomes completely deleted.
    text = re.sub(r"^.*?\.{10,}[\s\d]+$", "", text, flags=re.MULTILINE)


    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [8]:
def prepare_vector_database(pdf_dir=PDF_DOCS_DIR, local_db_dir=LOCAL_DB_DIR, drive_db_dir=DRIVE_DB_DIR, model_name=EMBEDDING_MODEL_NAME):
    print("--- Phase 0: Data Ingestion Started ---")

    if os.path.exists(local_db_dir):
        print(f"[*] Deleting old database: {local_db_dir}")
        shutil.rmtree(local_db_dir)

    if not os.path.exists(pdf_dir) or len(os.listdir(pdf_dir)) == 0:
        print(f"[!] ERROR: Folder '{pdf_dir}' not found or empty.")
        return

    print("[*] Loading PDFs using Unstructured (fast strategy)...")
    loader = DirectoryLoader(
        pdf_dir,
        glob="**/*.pdf",
        loader_cls=UnstructuredPDFLoader,
        loader_kwargs={"strategy": "fast", "mode": "single"}
    )
    documents = loader.load()

    if not documents:
        print("[!] No PDF documents could be read. Aborting.")
        return

    # 1. DATA CLEANING (THE HYBRID STEP)
    print("[*] Applying Regex filters to remove boilerplate text and noise...")
    for doc in documents:
        doc.page_content = clean_boilerplate_text(doc.page_content)

    print("[*] PDFs successfully processed and cleaned.")

    # 2. SEMANTIC CHUNKING
    print(f"[*] Initializing OpenAI Embedding Model: '{model_name}'")
    try:
        embeddings = OpenAIEmbeddings(model=model_name)
    except Exception as e:
        print(f"[!] Embedding model initialization failed: {e}")
        return

    print("[*] Performing semantic chunking on cleaned documents...")
    text_splitter = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile"
    )

    chunks = text_splitter.split_documents(documents)

    # 3. APPLY LOW VALUE SECTION FILTERS
    print("[*] Filtering out low-value chunks (Contents, Responsibilities, etc.)...")
    valid_chunks = []
    for chunk in chunks:
        content_lower = chunk.page_content.lower()

        # Check if chunk contains any low value section keywords
        is_low_value = any(keyword in content_lower for keyword in LOW_VALUE_KEYWORDS)

        # Skip chunks that are too short or contain low value patterns
        if len(chunk.page_content) > 100 and not is_low_value:
            valid_chunks.append(chunk)

    print(f"[*] Successfully split into {len(valid_chunks)} valid semantic chunks.")

    # 4. SAVE TO VECTOR DATABASE (Locally first)
    print(f"[*] Generating vectors and saving to ChromaDB: {local_db_dir}")
    vector_db = Chroma.from_documents(
        documents=valid_chunks,
        embedding=embeddings,
        persist_directory=local_db_dir
    )

    # 5. BACKUP TO GOOGLE DRIVE
    print(f"[*] Backing up compiled database to Google Drive: {drive_db_dir}")
    if os.path.exists(drive_db_dir):
        shutil.rmtree(drive_db_dir)
    shutil.copytree(local_db_dir, drive_db_dir)

    print("--- Data Ingestion Complete! ---")
    print(f"[*] Vector DB successfully built and backed up to Drive.")

In [9]:
# Execute pipeline
prepare_vector_database()

--- Phase 0: Data Ingestion Started ---
[*] Loading PDFs using Unstructured (fast strategy)...


[*] Applying Regex filters to remove boilerplate text and noise...
[*] PDFs successfully processed and cleaned.
[*] Initializing OpenAI Embedding Model: 'text-embedding-3-small'
[*] Performing semantic chunking on cleaned documents...
[*] Filtering out low-value chunks (Contents, Responsibilities, etc.)...
[*] Successfully split into 134 valid semantic chunks.
[*] Generating vectors and saving to ChromaDB: /content/local_vector_db/methodology_db
[*] Backing up compiled database to Google Drive: /content/drive/MyDrive/nice_rag_prod/vector_db/methodology_db
--- Data Ingestion Complete! ---
[*] Vector DB successfully built and backed up to Drive.


In [12]:
def inspect_database():
    """Loads ChromaDB, counts chunks, and runs a test semantic search."""
    print("\n[*] Initializing database inspection...")

    if not os.path.exists(LOCAL_DB_DIR):
        print(f"[!] ERROR: Database '{LOCAL_DB_DIR}' not found.")
        return

    print(f"[*] Loading embedding model: '{EMBEDDING_MODEL_NAME}'")
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)

    print(f"[*] Loading ChromaDB from: {LOCAL_DB_DIR}")
    vector_db = Chroma(
        persist_directory=LOCAL_DB_DIR,
        embedding_function=embeddings
    )

    total_chunks = vector_db._collection.count()
    print("\n================ DATABASE INFO ================")
    print(f"Total Semantic Chunks Stored: {total_chunks}")
    print("===============================================\n")

    sample_query = "Definitions, diagnostic criteria, and clinical management rules for Type 2 Diabetes and Obesity"

    # "What is the recommended step 1 antihypertensive drug treatment and specific clinic blood pressure target for an adult diagnosed with both Type 2 diabetes and primary hypertension?"

    # "How is hypertension diagnosed in adults, and what are the specific blood pressure thresholds for clinic and ambulatory readings?"

    # "What are the diagnostic criteria and thresholds for Type 2 Diabetes, including HbA1c levels?"

    print(f"[*] Testing semantic similarity search for:")
    print(f"    '{sample_query}'\n")

    results = vector_db.similarity_search_with_score(sample_query, k=10)

    if not results:
        print("[!] No relevant results found.")
        return

    print("================ SEARCH RESULTS ================")
    for index, (document, score) in enumerate(results, start=1):
        print(f"--- Result {index} (Distance: {score:.4f}) ---")
        source_file = document.metadata.get('source', 'Unknown Source')
        file_name = os.path.basename(source_file)
        print(f"Source PDF: {file_name}")

        content = document.page_content.replace('\n', ' ')
        print(f"Snippet:\n{content[:1000]}...\n")

In [13]:
# Run inspection
inspect_database()


[*] Initializing database inspection...
[*] Loading embedding model: 'text-embedding-3-small'
[*] Loading ChromaDB from: /content/local_vector_db/methodology_db

================ DATABASE INFO ================
Total Semantic Chunks Stored: 134

[*] Testing semantic similarity search for:
    'Definitions, diagnostic criteria, and clinical management rules for Type 2 Diabetes and Obesity'

================ SEARCH RESULTS ================
--- Result 1 (Distance: 0.9064) ---
Source PDF: qof_combined.pdf
Snippet:
27 Strain et al. Type 2 diabetes mellitus in older people: a brief statement of key principles of modern day management including the assessment of frailty. Diabetic medicine. 2018;35(7): 838-845. 28 NICE NG17 (2015, updated 2022). Type 1 diabetes in adults. https://www.nice.org.uk/guidance/NG17 29 NICE NG28 (2015, updated 2022). Type 2 diabetes in adults. https://www.nice.org.uk/guidance/NG28  39  DM034 (based on NICE IND275)  DM034 Rationale  i. Cardiovascular risk is elevated 